# [![](https://img.shields.io/badge/Kohya%20Setting-Wiki-Blue)](https://github.com/kohya-ss/sd-scripts/tree/sd3?tab=readme-ov-file#flux1-fine-tuning) [![](https://img.shields.io/badge/Video-Hướng%20dẫn-ff0000)](https://youtu.be/UCLlbxQD0uk) [![](https://img.shields.io/badge/Design-stablediffusion.vn-0075ff)](https://stablediffusion.vn) [![](https://img.shields.io/badge/Ver-1.0-0075ff)](https://trainlora.vn) [![](https://img.shields.io/badge/All%20Tools-sdvn.me-0075ff)](https://stablediffusion.vn/bo-cong-cu/) [![](https://img.shields.io/badge/Khoá%20học-SD%20Flux-red)](https://hungdiffusion.com/) [![](https://img.shields.io/badge/Group-Support-0075ff)](https://www.facebook.com/groups/stablediffusion.vn) [![](https://img.shields.io/discord/813085864355037235?color=blue&label=Discord&logo=Discord)](https://discord.gg/5SEtApPeyG)
---
`✨ Support Flux, Flux Krea, SD15, SDXL, all finetune model`

`🟡 Fix florence caption`

#💽 Install

In [ ]:
# @title ###💽 Download + Install
import os
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/SD-Data"
repo_dir = "/content/SDVN-training-colab-flux"

TrainType = "SDXL" # @param ["Flux", "SDXL", "SD15"]

brand_d = {
    "Flux": "sd3",
    "SDXL": "main",
    "SD15": "main"
}

model_folder = "/content/model"

#Install
!git clone -b dev https://github.com/StableDiffusionVN/SDVN-training-colab-flux
!pip install -q -r /content/SDVN-training-colab-flux/requirements.txt

if os.path.exists('/content/sd-scripts'):
  %cd /content/sd-scripts
  !git checkout {brand_d[TrainType]}
else:
  %cd /content
  !git clone -b {brand_d[TrainType]} https://github.com/kohya-ss/sd-scripts
  !sed -i 's/^bitsandbytes==0.44.0/bitsandbytes/' /content/sd-scripts/requirements.txt
if TrainType != "Flux":
  !sed -i 's/^# for Image utils/numpy<2.0/' /content/sd-scripts/requirements.txt

%cd /content/sd-scripts
!pip install -q -r /content/sd-scripts/requirements.txt
%cd /content
%run {repo_dir}/TrainScript.ipynb

#📂 II. Xử lý dữ liệu

In [ ]:
#@title ✨ 2.1 Caption

TrainFolder = "/content/drive/MyDrive/SD-Data/TrainData/PONY/HOG_Aqua_Ganyu_Sailor_Suit_46"  # @param {type:'string'}
DataClean = False #@param {type:"boolean"}
No_gen_caption = False #@param {type:"boolean"}
# @markdown 🟡 `OCR Prompt`

#@markdown 💡 ` Florence yêu cầu khởi động lại phiên (Ctr + M + .) và chạy lại từ đầu, nên dùng Gemini API thay thế hoặc dùng Caption_tools riêng`

#@markdown [![](https://img.shields.io/badge/Caption-Tools-blue)](https://colab.research.google.com/github/StableDiffusionVN/SDVN-training-colab-flux/blob/main/SDVN_Caption_Tools.ipynb)

Caption = 'WD14' # @param ['None', 'WD14','Florence', 'APIGemini | 2.5 Flash', ' APIGemini | 2.5 Pro', 'APIGemini | 2.5 Flash Lite',  'APIOpenAI | GPT 5', 'APIOpenAI | GPT 5-mini', 'APIOpenAI | GPT 5-nano']
Caption_Length = "Long" # @param ["Short","Medium","Long"]
# @markdown 🟡 `API Caption`
APIkey = "" # @param {type:'string'}
API_Prompt = "" # @param {type:'string'}
# @markdown 🟡 `Thêm caption tuỳ chọn`
Custom_Caption = "HOG_Aqua_Ganyu_Sailor_Suit" # @param {type:'string'}
AddFolderName = False #@param {type:"boolean"}
Remove_Caption = False #@param {type:"boolean"}
Append = False #@param {type:"boolean"}

if No_gen_caption == True:
  Caption = 'None'
  Custom_Caption = ''
  AddFolderName = False

Cap_prompt = {
    'Short':['<CAPTION>',0.85, 30],
    'Medium':['<DETAILED_CAPTION>',0.5, 60],
    'Long':['<MORE_DETAILED_CAPTION>',0.35, 100]
}
extension =".txt"

if DataClean == True :
  %cd /content
  clean_directory(TrainFolder)
if Caption != "None":
  caption_dir(TrainFolder,Cap_prompt[Caption_Length])
if AddFolderName:
  add_forder_name(TrainFolder)
if Custom_Caption != "":
  process_dir(TrainFolder, Custom_Caption, Append, Remove_Caption)


#⏳ III. Training Lora

In [ ]:
#@title ###⚡️ Cấu hình train Lora
%run {repo_dir}/Config.ipynb

output_name = "HOG_Aqua_Ganyu_Sailor_Suit" # @param {"type":"string"}
output_dir = "/content/drive/MyDrive/SD-Data/Lora/Pony/HOG/" # @param {"type":"string"}

#@markdown `💡 Nhận đường dẫn thư mục, đường dẫn tải về, địa chỉ civitAI, huggingface, danh sách trong list có sẵn. Mặc định Flux dùng model gốc`
model_train = "/content/drive/MyDrive/SD-Data/Model/Pony/harukiMIX_ponyV23.safetensors" #@param ["", "Flux-Krea-Dev", "Flux-Dev", "RealisticVision51", "SDXL-Base",  "STOIQO-NewRealityXLPro", "RealVisXLv5", "CharacterArtXL", "RealMythXL_v7", "XXMix_9realisticSDXL", "LEOSAM-HelloWorldXL", "AdamXL-v3", "JuggernautXL-v9", "Illustrious-XL", "PonyDiffusionV6XL"] {allow-input: true}
image_dir = TrainFolder
#@markdown `💡 Nhập nhiều size để train nhiều kích thước (chỉ dùng cho flux), vd: 768,1024,1280; chỉ định batch riêng cho từng size, vd: 768-6,1024-4,1280-2`
resolution = "1024" # @param {"type":"string"}
#@markdown `💡 Điều chỉnh theo mức độ tiêu thụ GPU(phụ thuộc GPU, resolution, optimizer_type)`
Batch_size = 16 # @param {"type":"number"}
flip = False # @param {"type":"boolean"}
color_aug = False # @param {"type":"boolean"}
shuffle_caption = False # @param {"type":"boolean"}
#@markdown `💡 Có thể điều chỉnh repeat cho từng thư mục bằng cách đặt tên thư mục theo cú pháp {repeat}_{folder_name}`
Max_train_steps = 5 # @param {"type":"number"}
Repeats = 5 # @param {"type":"number"}
Epochs = 10 # @param {"type":"number"}
Save_Every_N_Epochs = 2 # @param {"type":"number"}
Save_Every_N_Steps = 143 # @param {"type":"number"}
lr = 1e-4 # @param {"type":"number"}
text_lr = 5e-5 # @param {"type":"number"}
train_textencode = True # @param {"type":"boolean"}
lr_scheduler = "cosine" # @param ["constant","cosine","linear","cosine_with_restarts","adafactor","polynomial","constant_with_warmup"]
dim = 32 # @param {"type":"number"}
alpha = 32 # @param {"type":"number"}
optimizer_type = 'adamw8bit' # @param ["adafactor","adamw8bit"]

#@markdown `*️⃣ Tuỳ chọn mở rộng`
Extra = False # @param {"type":"boolean"}
author = '' # @param {"type":"string"}
#@markdown `💡 Gen ảnh sample hiện không khả dụng với SD15`
sample_prompt = '' # @param {"type":"string"}
sample_size = '768,1024' # @param {"type":"string"}
sample_steps = 100 # @param {"type":"number"}
save_state = False # @param {"type":"boolean"}
resume = '' # @param {"type":"string"}
wandbapi = '' # @param {"type":"string"}
#@markdown `☕️ Training`
runcode = True # @param {"type":"boolean"}
output_dir = f"{output_dir}/{output_name}"
dataset = f'/content/dataset_{output_name}.toml'
AutoDisconect = False # @param {"type":"boolean"}

#Config
if TrainType == 'SD15':
  sample_steps =0

model_train = download_lib(model_train)
data_config = data_config(flip, color_aug, shuffle_caption)
config = config(TrainType, model_train, clip_l_path, t5xxl_fp16_path, vae_path, dim, alpha, train_textencode, optimizer_type, lr, text_lr, lr_scheduler, Epochs, Save_Every_N_Epochs, Save_Every_N_Steps, dataset, output_dir, output_name)
extra = extra(TrainType, output_name, author,save_state, resume, wandbapi, sample_steps, runcode)
prompt(sample_prompt, sample_size, TrainFolder)
dataset_file(resolution, Batch_size, image_dir, Repeats, data_config, dataset)
arg = f'accelerate launch {dic2arg(op)} {lora_train_py[TrainType]} {dic2arg(config)} {dic2arg(extra) if Extra else ""}'
check_folder_train(image_dir)

if resume != "":
  !pip install -q -U accelerate

print('Chạy lệnh sau trong Terminal( Cửa sổ dòng lệnh) hoặc bật chế độ runcode (Sẽ không hiển thị thông báo):')
print(f"cd /content/sd-scripts;{arg}")
if runcode:
  %cd /content/sd-scripts
  !{arg}
display(HTML('<div id="finish_train"><h3>Kết thúc train</h3></div>'))
if AutoDisconect:
  from time import sleep
  sleep(3*60)
  from google.colab import runtime
  runtime.unassign()